# Week 31, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w031d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Crate:
    def __init__(self, n):
        self.n = n
    def __len__(self):
        return self.n

class Lift:
    def __index__(self):
        return 1

print(bool(Crate(0)), bool(Crate(4)), ["G", "1st", "2nd"][Lift()])

**Predict before you run:** what will this print?



In [ ]:
class Grid:
    def __getitem__(self, key):
        return type(key).__name__

class Count:
    def __getitem__(self, i):
        if i >= 3:
            raise IndexError
        return i * 10

g = Grid()
print(g[1], g[1, 2], g[1:2], list(Count()), 20 in Count())

**Predict before you run:** what will this print?



In [ ]:
from functools import total_ordering

@total_ordering
class Size:
    ORDER = ["S", "M", "L"]
    def __init__(self, s):
        self.s = s
    def __eq__(self, other):
        return self.s == other.s
    def __lt__(self, other):
        return Size.ORDER.index(self.s) < Size.ORDER.index(other.s)

print(Size("L") > Size("M"), Size("S") >= Size("M"), max([Size("M"), Size("L"), Size("S")]).s)

**Predict before you run:** what will this print?



In [ ]:
class Stamp:
    def __init__(self):
        self.n = 0
    def __call__(self, text):
        self.n += 1
        return f"{self.n}. {text}"

s = Stamp()
print([s(x) for x in "ab"], s("c"), callable(s), callable(s.n))

**Predict before you run:** what will this print?



In [ ]:
class A:
    __slots__ = ("x",)

class B(A):
    __slots__ = ()

class C(A):
    pass

print(hasattr(B(), "__dict__"), hasattr(C(), "__dict__"))

### Exercise: A bag of things

Write `Bag(items=())`, which counts how many of each thing it holds (like Week 20's `Counter`, but your own):

- `bag[item]` is how many of that item there are, and **0** for an item it doesn't hold (a bag is a mapping that never raises `KeyError`);
- `bag.add(item, n=1)` adds `n` (at least 1, otherwise `ValueError`);
- `len(bag)` is the total number of things, `item in bag` is True if there's at least one, and iterating gives each **distinct** item once, in the order first added;
- `a + b` is a new bag holding both; `==` compares the counts.

In [ ]:
class Bag:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bag():\n    \"bags\"\n    B = ns[\"Bag\"]\n    b = B([\"egg\", \"egg\", \"yam\"])\n    b.add(\"egg\")\n    b.add(\"rice\", 3)\n    assert b[\"egg\"] == 3 and b[\"yam\"] == 1 and b[\"plantain\"] == 0 and len(b) == 7\n    assert \"rice\" in b and \"plantain\" not in b and list(b) == [\"egg\", \"yam\", \"rice\"]\n    c = b + B([\"yam\", \"salt\"])\n    assert c[\"yam\"] == 2 and len(c) == 9 and b[\"yam\"] == 1 and list(c) == [\"egg\", \"yam\", \"rice\", \"salt\"]\n    assert B([\"a\", \"b\", \"a\"]) == B([\"b\", \"a\", \"a\"]) and B([\"a\"]) != B([\"a\", \"a\"]) and not B()\n    try:\n        b.add(\"x\", 0)\n        assert False, \"n below 1 should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Ordered temperatures

Write `Celsius(degrees)`, fully ordered and hashable (use `total_ordering`), with `NotImplemented` for anything that isn't a `Celsius`. `str` gives `21.5°C`. `__format__` supports `"F"` (Fahrenheit to 1 place, like `70.7°F`), an empty spec (the `str`), and hands any other spec to the number of degrees, so `f"{t:.0f}"` gives `22`.

In [ ]:
from functools import total_ordering

class Celsius:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_temps():\n    \"temperatures\"\n    C = ns[\"Celsius\"]\n    t = C(21.5)\n    assert str(t) == \"21.5\u00b0C\" and f\"{t}\" == \"21.5\u00b0C\" and f\"{t:F}\" == \"70.7\u00b0F\" and f\"{t:.0f}\" == \"22\" and f\"{C(-40):F}\" == \"-40.0\u00b0F\"\n    assert C(5) < C(10) <= C(10) and C(30) > C(-3) and max([C(1), C(30), C(4)]) == C(30) and len({C(2), C(2)}) == 1\n    try:\n        C(5) < 6\n        assert False, \"comparing with a number should raise TypeError\"\n    except TypeError:\n        pass")

### Exercise: Forge: a matrix class

Write `Matrix` to the specification above.

In [ ]:
class Matrix:
    __slots__ = ("_rows",)
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def M(rows):\n    return ns[\"Matrix\"](rows)\ndef test_make():\n    \"making and looking\"\n    rows = [[1, 2, 3], [4, 5, 6]]\n    m = M(rows)\n    rows[0][0] = 99\n    assert m.shape == (2, 3) and len(m) == 2 and m[0, 0] == 1 and m[1, 2] == 6 and m[-1, -1] == 6\n    assert m[1] == (4, 5, 6) and list(m) == [(1, 2, 3), (4, 5, 6)] and 5 in m and 99 not in m\n    assert repr(m) == \"Matrix([[1, 2, 3], [4, 5, 6]])\" and not hasattr(m, \"__dict__\")\n    assert str(M([[1, 20], [300, 4]])) == \"  1  20\\n300   4\", f\"got {str(M([[1, 20], [300, 4]]))!r}\"\n    assert str(M([[1.5]])) == \"1.5\" and M(((1, 2), (3, 4)))[1, 0] == 3\n    I = ns[\"Matrix\"].identity(3)\n    assert list(I) == [(1, 0, 0), (0, 1, 0), (0, 0, 1)]\n    for bad in [[], [[]], [[1, 2], [3]], [[1, \"2\"]], [[True]]]:\n        try:\n            M(bad)\n            assert False, f\"Matrix({bad!r}) should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        m.shape = (1, 1)\n        assert False, \"shape should be read-only\"\n    except AttributeError:\n        pass\ndef test_setting():\n    \"changing entries\"\n    m = M([[1, 2], [3, 4]])\n    m[0, 1] = 7.5\n    m[-1, 0] = 0\n    assert list(m) == [(1, 7.5), (0, 4)]\n    for key in [(2, 0), (0, 2), (-3, 0)]:\n        for action in (\"get\", \"set\"):\n            try:\n                if action == \"get\":\n                    m[key]\n                else:\n                    m[key] = 1\n                assert False, f\"m[{key}] should raise IndexError\"\n            except IndexError:\n                pass\n    try:\n        m[0, 0] = \"x\"\n        assert False, \"a non-number should raise ValueError\"\n    except ValueError:\n        pass\n    try:\n        m[0] = [9, 9]\n        assert False, \"replacing a whole row should raise TypeError\"\n    except TypeError:\n        pass\n    try:\n        m[5]\n        assert False, \"m[5] should raise IndexError\"\n    except IndexError:\n        pass\ndef test_arithmetic():\n    \"arithmetic\"\n    a, b = M([[1, 2], [3, 4]]), M([[5, 6], [7, 8]])\n    assert a + b == M([[6, 8], [10, 12]]) and b - a == M([[4, 4], [4, 4]]) and a * 2 == M([[2, 4], [6, 8]]) and 3 * a == a * 3\n    assert a @ b == M([[19, 22], [43, 50]]) and a @ ns[\"Matrix\"].identity(2) == a\n    assert M([[1, 2, 3]]) @ M([[1], [2], [3]]) == M([[14]]) and (M([[1], [2]]) @ M([[3, 4]])).shape == (2, 2)\n    assert a == M([[1, 2], [3, 4]]) and a != M([[1, 2]]) and a != [[1, 2], [3, 4]] and a.__eq__(5) is NotImplemented\n    assert a == M([[1, 2], [3, 4]]) and b == M([[5, 6], [7, 8]]), \"operations shouldn't change their operands\"\n    for f in [lambda: a + M([[1, 2]]), lambda: a @ M([[1, 2, 3]])]:\n        try:\n            f()\n            assert False, \"mismatched shapes should raise ValueError\"\n        except ValueError:\n            pass\n    for f in [lambda: a + 1, lambda: a * a, lambda: a @ 2, lambda: \"x\" * a, lambda: hash(a)]:\n        try:\n            f()\n            assert False, \"should raise TypeError\"\n        except TypeError:\n            pass\ndef test_methods():\n    \"transpose and apply\"\n    m = M([[1, 2, 3], [4, 5, 6]])\n    t = m.transpose()\n    assert list(t) == [(1, 4), (2, 5), (3, 6)] and m.shape == (2, 3)\n    assert m.apply(lambda x: x * x) == M([[1, 4, 9], [16, 25, 36]]) and list(M([[1.26]]).apply(round)) == [(1,)]\n    class Plus:\n        def __init__(self, n):\n            self.n = n\n        def __call__(self, x):\n            return x + self.n\n    assert m.apply(Plus(10))[0, 0] == 11")

> **Special methods are a promise.** Look at how much of `Matrix` is special methods, and how little of it callers have to learn: `m[i, j]`, `for row in m`, `5 in m`, `a @ b`, `print(m)`. Each one keeps a promise that Python's own types make, so anyone who knows lists and numbers can use your matrix without reading its code. That's the real reason for the data model. It also means each method should behave the way its built-in cousins do: raise `IndexError` and `KeyError` where they would, return `NotImplemented` for unknown types, and never change the operands of `+`. A special method that breaks those expectations is worse than an ordinary method with a clear name.

### Exercise: Forge bonus: a least-recently-used cache

Write `LRUCache(capacity)` (capacity at least 1, otherwise `ValueError`), a mapping that holds at most `capacity` items:

- `cache[key] = value` adds or updates an item and makes it the **most** recently used. If that takes the cache over capacity, the least recently used item is removed, and its key is appended to a list `evicted`.
- `cache[key]` returns the value and makes it the most recently used; a missing key raises `KeyError`.
- `key in cache` checks without counting as a use. `len(cache)` is the number of items. `del cache[key]` removes an item (not counted in `evicted`).
- Iterating gives the keys from **least** to **most** recently used.

Then write `Cached(func, capacity)`, a callable wrapper that remembers `func`'s results by their positional arguments in an `LRUCache`, with `hits` and `misses` counts and the cache itself as `cache`.

In [ ]:
class LRUCache:
    ___

class Cached:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"the cache\"\n    L = ns[\"LRUCache\"]\n    c = L(2)\n    c[\"a\"] = 1\n    c[\"b\"] = 2\n    assert c[\"a\"] == 1, \"reading a makes it most recent\"\n    c[\"c\"] = 3\n    assert list(c) == [\"a\", \"c\"] and c.evicted == [\"b\"] and \"b\" not in c and len(c) == 2\n    c[\"a\"] = 10\n    c[\"d\"] = 4\n    assert list(c) == [\"a\", \"d\"] and c.evicted == [\"b\", \"c\"] and c[\"a\"] == 10\n    assert \"d\" in c and list(c) == [\"d\", \"a\"], \"in shouldn't count as a use\"\n    del c[\"d\"]\n    assert list(c) == [\"a\"] and c.evicted == [\"b\", \"c\"]\n    try:\n        c[\"zzz\"]\n        assert False, \"a missing key should raise KeyError\"\n    except KeyError:\n        pass\n    try:\n        L(0)\n        assert False, \"capacity 0 should raise ValueError\"\n    except ValueError:\n        pass\ndef test_cached():\n    \"the wrapper\"\n    Cd = ns[\"Cached\"]\n    calls = []\n    def slow_square(x):\n        calls.append(x)\n        return x * x\n    sq = Cd(slow_square, 2)\n    assert [sq(2), sq(3), sq(2), sq(4), sq(3)] == [4, 9, 4, 16, 9]\n    assert calls == [2, 3, 4, 3], f\"calls were {calls}\"\n    assert (sq.hits, sq.misses) == (1, 4) and list(sq.cache) == [(4,), (3,)] and sq.cache.evicted == [(3,), (2,)]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [the tour](w031d1.html), [container protocols](w031d2.html), [rich comparisons](w031d3.html), [callable objects](w031d4.html) or [slots](w031d5.html).

## Mixed review

## Forge: a matrix class

A **matrix** is a rectangle of numbers, written in rows and columns. Matrices are used everywhere numbers come in tables: turning and scaling shapes on a screen, solving several equations at once, and (on a much bigger scale) in machine learning. Python has an operator kept especially for them: `a @ b`, which calls `__matmul__`, for **matrix multiplication**. To multiply `a` (with `n` columns) by `b` (with `n` rows), each entry of the result, in row `i` and column `j`, is the sum of row `i` of `a` times column `j` of `b`, entry by entry:

```text
[1 2]   [5 6]   [1*5 + 2*7   1*6 + 2*8]   [19 22]
[3 4] @ [7 8] = [3*5 + 4*7   3*6 + 4*8] = [43 50]
```

Here's the specification for `Matrix`.

**Making one**

- `Matrix(rows)` takes a list (or any iterable) of rows, each a list (or any iterable) of numbers. There must be at least one row, every row must have the same length (at least 1), and every entry must be an `int` or `float` (not a `bool`); otherwise `ValueError`. It keeps its own copy, so changing the caller's lists afterwards doesn't change the matrix.
- `Matrix.identity(n)` is a class method giving the `n` by `n` matrix with 1 on the diagonal and 0 elsewhere (`n` at least 1).
- The class uses `__slots__`, so matrices have no `__dict__`.

**Looking at it**

- `m.shape` is a read-only property: `(rows, columns)`.
- `len(m)` is the number of rows. Iterating gives the rows as **tuples**.
- `m[i]` gives row `i` as a tuple; `m[i, j]` gives one entry. Negative positions count from the end, as in lists; positions out of range raise `IndexError`.
- `m[i, j] = value` changes one entry (`IndexError` out of range; `ValueError` if `value` isn't a number as above). `m[i] = ...` isn't allowed: raise `TypeError`.
- `value in m` is True if any entry equals it.
- `repr(m)` gives `Matrix([[1, 2], [3, 4]])`. `str(m)` gives one line per row, each entry right-aligned to the width of the widest entry in the whole matrix (using `str` of each entry), separated by single spaces: `Matrix([[1, 20], [300, 4]])` prints as `  1  20` and `300   4`.

**Arithmetic**

- `m == n` is True when the shapes and all entries match (`NotImplemented` for non-matrices). Matrices can change, so they must be unhashable.
- `m + n` and `m - n` work entry by entry and need the same shape (`ValueError` otherwise).
- `m * k` and `k * m` multiply every entry by a number.
- `m @ n` is the matrix product; `m`'s column count must equal `n`'s row count (`ValueError` otherwise).
- Anything else (`m + 1`, `m * m`, `m @ 2`) raises `TypeError`, through `NotImplemented`.
- Every operation returns a **new** matrix and leaves its operands unchanged.

**Two more methods**

- `transpose()` returns a new matrix with rows and columns swapped.
- `apply(func)` returns a new matrix with `func` (any callable) applied to every entry.

## Forge bonus: a least-recently-used cache

A cache keeps answers that were expensive to work out, but memory is limited, so a cache usually has a **capacity**. When it's full and a new answer arrives, something must go. A common rule is to throw out the **least recently used** item: the one that hasn't been read or written for longest. Week 20 built one with `OrderedDict`; this time it's a class of your own that behaves like a dictionary. Python dictionaries keep their keys in the order they were added, so removing a key and adding it again moves it to the end.

## Weekly quiz

## Week 31 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain the data model, and write `__len__`, `__bool__`, conversions, `__format__` and `__getattr__`. ([Day 1](w031d1.html))
- I can write sequences and mappings with `__getitem__`, `__setitem__`, `__delitem__`, `__contains__` and `__iter__`. ([Day 2](w031d2.html))
- I can make a class ordered, with `total_ordering`, and know when to use a key function instead. ([Day 3](w031d3.html))
- I can write callable objects that keep settings and state, and wrap functions with them. ([Day 4](w031d4.html))
- I can use `__slots__` with inheritance, measure memory, and make objects immutable. ([Day 5](w031d5.html))
- I built a matrix class and a least-recently-used cache that behave like Python's own types. (Today)

Tomorrow is a **rest day**. Next week: **modern classes**, where dataclasses, Enums, NamedTuple, abstract base classes and Protocols write much of this week's code for you.